In [ ]:
import mne                       #MNE-Python 사용을 위해
import numpy as np               #python의 대규모 수학 계산을 도와줌
import pandas as pd              #데이터 분석을 도와줌

import seaborn as sns            #시각화 라이브러리
from matplotlib.colors import TwoSlopeNorm #색상 범위 조절

import scipy.io                  #Matlab 파일을 읽어오기 위하여
import addcopyfighandler         #figure를 복사하기 위하여
import matplotlib.pyplot as plt  #시각화 라이브러리

In [2]:
%matplotlib qt 
#그래프를 별도의 창에서 열기 위해서

In [3]:
# mat file 불러오기
mat = scipy.io.loadmat(r'\\HCN-NAS\ywYang\s14000.mat')

In [4]:
# mat 구조 살펴보기
for i in mat:
    print(i)
print()

__header__
__version__
__globals__
eeg



In [5]:
#eeg data 추출하기
fields = list(mat['eeg'][0].dtype.fields.keys())


In [6]:
for k in range(len(fields)):
    print(f"{fields[k]} 필드 데이터:")
    print("값:", mat['eeg'][0][fields[k]][0][0])
    print("길이:", len(mat['eeg'][0][fields[k]][0][0]))
    print()

raw_left 필드 데이터:
값: [-186856.25 -186927.75 -186951.   ... -158500.25 -158470.5  -158491.75]
길이: 358400

raw_right 필드 데이터:
값: [-185748.   -185525.75 -185612.5  ... -154575.25 -154814.25 -155262.75]
길이: 358400

n_trials 필드 데이터:
값: [100]
길이: 1

frame 필드 데이터:
값: [-2000  5000]
길이: 2

srate 필드 데이터:
값: [512]
길이: 1

event 필드 데이터:
값: [0 0 0 ... 0 0 0]
길이: 358400

senloc 필드 데이터:
값: [-2.1185074   8.85472836  6.05844269]
길이: 3

psenloc 필드 데이터:
값: [-0.19371631  0.80967632  0.55398397]
길이: 3

subject 필드 데이터:
값: s14000
길이: 6

comment 필드 데이터:
값: bci2011v1 left/right imagery experiment
길이: 39

emg_left 필드 데이터:
값: [-350009.25 -349878.5  -350106.75 ... -278221.   -278275.5  -278582.25]
길이: 358400

emg_right 필드 데이터:
값: [-351785.75 -351616.   -351778.   ... -278058.5  -277717.5  -277276.  ]
길이: 358400

rest 필드 데이터:
값: [-180233.   -180281.75 -180299.5  ... -182963.   -182764.25 -182684.75]
길이: 34048

noise 필드 데이터:
값: [array([[-173193.   , -173266.75 , -173374.25 , ..., -180139.   ,
         -180105.5  , -17

In [7]:
#eeg 채널 이름 정의
eeg_ch_names = [
    "Fp1", "AF7", "AF3", "F1", "F3", "F5", "F7", "FT7", "FC5","FC3", "FC1", 
    "C1", "C3", "C5", "T7", "TP7", "CP5", "CP3", "CP1", "P1", "P3", "P5", "P7", 
    "P9", "PO7", "PO3", "O1", "lz", "Oz", "POz", "Pz", "CPz", "FPz", "FP2", 
    "AF8", "AF4", "AFz", "Fz", "F2", "F4", "F6", "F8", "FT8", "FC6", "FC4",
    "FC2", "FCz", "Cz", "C2", "C4", "C6", "T8", "TP8", "CP6", "CP4", "CP2", 
    "P2", "P4", "P6", "P8", "P10", "PO8", "PO4", "O2",
]

In [8]:
#emg 채널 이름 정의
emg_ch_names = ["EMG1", "EMG2", "EMG3", "EMG4"]

In [9]:
#stim이라는 채널과 eeg, emg 채널들 결합
ch_names = eeg_ch_names + emg_ch_names + ["stim"]

In [10]:
#채널 유형 정의 -> eeg 채널 64개, emg 채널 4개, stim 채널 1개
ch_types = ["eeg"] * 64 + ["emg"] * 4 + ["stim"] * 1

In [11]:
#eeg 구조체 안에서 sampling rate(sfreq) 추출하기
#sampling_freq = mat['eeg'][0][fields[4]][0][0]
sampling_freq = 512 #Hz
print("Sampling_freq:", sampling_freq)

Sampling_freq: 512


In [12]:
#info 생성
info = mne.create_info(ch_names=ch_names, ch_types=ch_types, sfreq=sampling_freq)
info["description"] = "Dataset"
info["bads"] = []
print(info)

<Info | 8 non-empty values
 bads: []
 ch_names: Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3, FC1, C1, C3, C5, ...
 chs: 64 EEG, 4 EMG, 1 Stimulus
 custom_ref_applied: False
 description: Dataset
 highpass: 0.0 Hz
 lowpass: 256.0 Hz
 meas_date: unspecified
 nchan: 69
 projs: []
 sfreq: 512.0 Hz
>


In [13]:
left = np.vstack((mat['eeg'][0]['raw_left'][0], mat['eeg'][0]['emg_left'][0], mat['eeg'][0]['event'][0]))
right = np.vstack((mat['eeg'][0]['raw_right'][0], mat['eeg'][0]['emg_right'][0], mat['eeg'][0]['event'][0]))

In [14]:
lr = mne.io.RawArray(left, info=info)
rr = mne.io.RawArray(right, info=info)
raw = [lr, rr]

Creating RawArray with float64 data, n_channels=69, n_times=358400
    Range : 0 ... 358399 =      0.000 ...   699.998 secs
Ready.
Creating RawArray with float64 data, n_channels=69, n_times=358400
    Range : 0 ... 358399 =      0.000 ...   699.998 secs
Ready.


In [15]:
LEFT = 0
RIGHT = 1

In [16]:
mne.set_eeg_reference(raw[LEFT], ch_type='eeg')
mne.set_eeg_reference(raw[RIGHT], ch_type='eeg')

Applying average reference.
Applying a custom ('EEG',) reference.
Applying average reference.
Applying a custom ('EEG',) reference.


(<RawArray | 69 x 358400 (700.0 s), ~188.7 MiB, data loaded>,
 array([-44639.93304443, -44534.70257568, -44691.90185547, ...,
        -42895.23364258, -42959.86254883, -43114.57739258], shape=(358400,)))

In [17]:
print("Raw의 길이:", len(raw[LEFT])) # 358400
print("초로 환산하면:", len(raw[LEFT]) / sampling_freq) # 700초
print("trials로 나누면:", len(raw[LEFT]) / sampling_freq / mat['eeg'][0]['n_trials'][0][0]) # trial 하나에7초

Raw의 길이: 358400
초로 환산하면: 700.0
trials로 나누면: [7.]


In [18]:
#events 추출하기
le = mne.find_events(raw[LEFT])
re = mne.find_events(raw[RIGHT])
events = [le, re]

Finding events on: stim
100 events found on stim channel stim
Event IDs: [1]
Finding events on: stim
100 events found on stim channel stim
Event IDs: [1]


In [19]:
raw_data = raw[LEFT].get_data()
plt.plot(raw_data[0])
plt.show()

In [20]:
#Epochs 만들기
le = mne.Epochs(raw[LEFT],
                events[LEFT], 
                tmin = -2, tmax = 5, 
                picks = ("C3", "C4"),
                baseline = (-0.5, 0),
                preload=True,)
re = mne.Epochs(raw[RIGHT],
                events[RIGHT],
                tmin = -2, tmax = 5,
                picks = ["C3", "C4"],
                baseline = (-0.5, 0),
                preload = True,)
epochs = [le, re]

Not setting metadata
100 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 100 events and 3585 original time points ...
1 bad epochs dropped
Not setting metadata
100 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 100 events and 3585 original time points ...
1 bad epochs dropped


In [22]:
le2 = mne.Epochs(raw[LEFT],
                 events[LEFT],
                 tmin = -2, tmax = 5,
                 picks = ("C3", "C4"),
                 baseline = None,
                 preload=True,)
re2 = mne.Epochs(raw[RIGHT],
                 events[RIGHT],
                 tmin = -2, tmax = 5,
                 picks = ("C3", "C4"),
                 baseline = None,
                 preload=True,)
epochs = [le2, re2]

Not setting metadata
100 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 100 events and 3585 original time points ...
1 bad epochs dropped
Not setting metadata
100 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 100 events and 3585 original time points ...
1 bad epochs dropped


In [23]:
C3 = 0
C4= 1

In [25]:
print(epochs[LEFT].get_data().shape)
print(epochs[RIGHT].get_data().shape)

(99, 2, 3585)
(99, 2, 3585)


In [26]:
data = [epochs[LEFT].get_data(), epochs[RIGHT].get_data()]

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(8, 8))

axes[0].plot(data[LEFT][0][C3], label='C3')
axes[0].plot(data[LEFT][0][C4], label='C4')
axes[0].set_title("Left")
axes[0].legend()

axes[1].plot(data[RIGHT][0][C3], label='C3')
axes[1].plot(data[RIGHT][0][C4], label='C4')
axes[1].set_title("Right")
axes[1].legend()

plt.show()